# Single-Handed Sign Language Recognition and Translation --Sample Trial

## Dataset 1 (from Github)

In [ ]:
# Importing the Necessary Libraries

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

import joblib

In [ ]:
from sklearn.preprocessing import StandardScaler, LabelEncoder
from scipy.signal import savgol_filter

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

import warnings
warnings.filterwarnings("ignore")

In [ ]:
# Working on the temporary dataset 1 (Restricted to LABEL ∈ [1, 19])
# Loading the dataset
data = pd.read_csv("sample_trial_dataset.csv")
print("Priginal dataset shape:", data.shape)

data.head(10)
#data.tail(10)

Priginal dataset shape: (13296, 9)


,THUMB,INDEX,MIDDLE,RING,LITTLE,G1,G2,G3,LABEL
0,666,732,700,965,969,15120,-736,6704,1
1,666,732,701,965,969,15208,-1344,6440,1
2,665,732,701,965,968,15264,-1724,6492,1
3,665,732,701,965,969,15444,-996,6540,1
4,665,732,701,965,968,15536,-1216,6436,1
5,665,732,701,965,968,15112,-796,6268,1
6,665,731,701,965,968,15224,-1096,6504,1
7,665,732,701,965,968,15140,-1104,6692,1
8,665,732,701,965,968,15104,-788,6384,1
9,665,732,701,965,968,15388,-972,5668,1


In [34]:
# Filtering valid labels (1-->19)
data = data[(data["LABEL"] >= 1) & (data["LABEL"] <= 19)]

print("The filtered dataset shape:", data.shape)
print("Remaining labels:", sorted(data["LABEL"].unique()))

The filtered dataset shape: (9287, 9)
Remaining labels: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18), np.int64(19)]


In [35]:
# Seperating the features and labels
X = data.drop(columns=["LABEL"])
y = data["LABEL"]

# Basic cleaning
# Removing duplicates
data = data.drop_duplicates()

# Forward-fill missing values (one way of solving it though not always recommanded)
#data = data.fillna(method="ffill")

In [40]:
# Noise filtering (sensor smoothing) --> suggested technique
X_smooth = pd.DataFrame(
    savgol_filter(X, window_length=7, polyorder=2, axis=0),
    columns = X.columns
)

# Normalization --> Suggested technique
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_smooth)

X_scaled = pd.DataFrame(X_scaled, columns=X.columns)

# Label Encoding (clean & consistent) --> This data is already encoded
'''label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

print("Encoded Classes:", label_encoder.classes_)'''

'label_encoder = LabelEncoder()\ny_encoded = label_encoder.fit_transform(y)\n\nprint("Encoded Classes:", label_encoder.classes_)'

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])

In [ ]:
X_scaled["LAEBEL"] = y.values

#X_scaled.to_csv("preprocessed_dataset_labels_1_19.csv", index=False)
#print("Preprocessed dataset saved successfully!")

In [ ]:
processed_df = pd.read_csv("preprocessed_dataset_labels_1_19.csv")

print(processed_df.head())
print(processed_df.info())

In [ ]:
label_counts = processed_df["LABEL"].value_counts().sort_index()
print(label_counts)

In [ ]:
label_counts.plot(kind="bar", figsize=(10,4))
plt.title("Distribution og Gesture Labels (1-19)")
plt.xlabel("Gesture Label")
plt.ylabel("Number of samples")
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)

y_pred = knn.predict(X_test)
print("Baseline KNN Accuracy:", accuracy_score(y_test, y_pred))

## Dataset 2 (from Kaggle)

In [ ]:
df1 = pd.read_csv("E:\Documents2\Semester 8\ENGI 402\MY_AI_CAP\sensors_data_1.csv")
df2 = pd.read_csv("E:\Documents2\Semester 8\ENGI 402\MY_AI_CAP\sensors_data_2.csv")
df3 = pd.read_csv("E:\Documents2\Semester 8\ENGI 402\MY_AI_CAP\sensors_data_4.csv")

print(df1.columns)
print(df2.columns)
print(df3.columns)

In [ ]:
# Flex Sensors (enabling for right hand only)

def aggregate_frames(df, prefix):
    cols = [col for col in df.columns if prefix in col]
    return df[cols].mean(axis=1)

df_clean = pd.DataFrame()

df_clean["FLEX_THUMB"]  = aggregate_frames(df1, "Flex-Right-1")
df_clean["FLEX_INDEX"]  = aggregate_frames(df1, "Flex-Right-2")
df_clean["FLEX_MIDDLE"] = aggregate_frames(df1, "Flex-Right-3")
df_clean["FLEX_RING"]   = aggregate_frames(df1, "Flex-Right-4")
df_clean["FLEX_LITTLE"] = aggregate_frames(df1, "Flex-Right-5")

df_clean["ACC_X"] = aggregate_frames(df1, "Orientation-X-Right")
df_clean["ACC_Y"] = aggregate_frames(df1, "Orientation-Y-Right")
df_clean["ACC_Z"] = aggregate_frames(df1, "Orientation-Z-Right")

df_clean["GYRO_X"] = aggregate_frames(df1, "Position-X-Right")
df_clean["GYRO_Y"] = aggregate_frames(df1, "Position-Y-Right")
df_clean["GYRO_Z"] = aggregate_frames(df1, "Position-Z-Right")

df_clean["SIGN"] = df1["SIGN"]

print(df_clean.head())
print("New shape:", df_clean.shape)


In [ ]:
# Now Applying the Same Process to the Remaining CSV Files

def process_file(path):
    df = pd.read_csv(path)
    df_clean = pd.DataFrame()

    df_clean["FLEX_THUMB"]  = aggregate_frames(df, "Flex-Right-1")
    df_clean["FLEX_INDEX"]  = aggregate_frames(df, "Flex-Right-2")
    df_clean["FLEX_MIDDLE"] = aggregate_frames(df, "Flex-Right-3")
    df_clean["FLEX_RING"]   = aggregate_frames(df, "Flex-Right-4")
    df_clean["FLEX_LITTLE"] = aggregate_frames(df, "Flex-Right-5")

    df_clean["ACC_X"] = aggregate_frames(df, "Orientation-X-Right")
    df_clean["ACC_Y"] = aggregate_frames(df, "Orientation-Y-Right")
    df_clean["ACC_Z"] = aggregate_frames(df, "Orientation-Z-Right")

    df_clean["GYRO_X"] = aggregate_frames(df, "Position-X-Right")
    df_clean["GYRO_Y"] = aggregate_frames(df, "Position-Y-Right")
    df_clean["GYRO_Z"] = aggregate_frames(df, "Position-Z-Right")

    df_clean["SIGN"] = df["SIGN"]

    return df_clean

df1 = process_file("E:\Documents2\Semester 8\ENGI 402\MY_AI_CAP\sensors_data_1.csv")
df2 = process_file("E:\Documents2\Semester 8\ENGI 402\MY_AI_CAP\sensors_data_2.csv")
df3 = process_file("E:\Documents2\Semester 8\ENGI 402\MY_AI_CAP\sensor_data_4.csv")


In [ ]:
# Now Merging Them Into One Dataset for Convenience

merged_df = pd.concat([df1, df2, df3], ignore_index=True)
merged_df.to_csv("final_clean_glove_dataset.csv", index=False)
print("Final dataset shape:", merged_df.shape)

In [ ]:
merged_df.info()

In [ ]:
print("Unique Signs:")
print(sorted(merged_df["SIGN"].unique()))

In [ ]:
print("Samples per Sign:")
print(merged_df["SIGN"].value_counts())

In [ ]:
# Cleaning Code for Dataset 2

print("Initial Shape:", merged_df.shape)
# Removing Duplicate Rows
merged_df = merged_df.drop_duplicates()
print("After removing duplicates:", merged_df.shape)

# Ensuring the Numeric Types
feature_columns = [
    "FLEX_THUMB", "FLEX_INDEX", "FLEX_MIDDLE",
    "FLEX_RING", "FLEX_LITTLE",
    "ACC_X", "ACC_Y", "ACC_Z", "GYRO_X", "GYRO_Y", "GYRO_Z"
]
# Converting to Numeric Safely Just in Case
for col in feature_columns:
    merged_df[col] = pd.to_numeric(merged_df[col], errors="coerce")

# Handling Missing Values (incase there are)
print("Missing values before cleaning:")
print(merged_df.isnull().sum())

# Option A: Removing rows with Missing Values
merged_df = merged_df.dropna()
# (Alternative Option if Needed)
# merged_df.fillna(merged_df.mean(), inplace=True)
print("Missing values after cleaning:")
print(merged_df.isnull().sum())

In [ ]:
# Outlier Handling (IQR METHOD)

def remove_outliers_iqr(df, columns):
    df_clean = df.copy()
    bounds = {}
    
    for col in columns:
        Q1 = df_clean[col].quantile(0.25)
        Q3 = df_clean[col].quantile(0.75)
        IQR = Q3 - Q1
        
        lower = Q1 - 1.5 * IQR
        upper = Q3 + 1.5 * IQR
        
        bounds[col] = {"lower": lower, "upper": upper}  # to capture the bounds

        df_clean = df_clean[(df_clean[col] >= lower) & (df_clean[col] <= upper)]
    
    # saving the bounds
    import json
    json.dump(bounds, open("test_iqr_bounds.json", "w"))

    return df_clean

merged_df = remove_outliers_iqr(merged_df, feature_columns)
print("After outlier removal:", merged_df.shape)

In [ ]:
# Feature Scaling

X = merged_df[feature_columns]
y = merged_df["SIGN"]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_scaled = pd.DataFrame(X_scaled, columns=feature_columns)

# saving scaler and feature columns
import joblib, json
joblib.dump(scaler, "test_scaler.pkl")
json.dump(feature_columns, open("test_feature_columns.json", "w"))

# Adding the Label Back
X_scaled["SIGN"] = y.values

In [ ]:
# Train Test Split

X_final = X_scaled.drop("SIGN", axis=1)
y_final = X_scaled["SIGN"]

X_train, X_test, y_train, y_test = train_test_split(
    X_final,
    y_final,
    test_size=0.2,
    random_state=42,
    stratify=y_final
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

# Saving the Results

'''X_train.to_csv("X_train.csv", index=False)
X_test.to_csv("X_test.csv", index=False)
y_train.to_csv("y_train.csv", index=False)
y_test.to_csv("y_test.csv", index=False)

X_scaled.to_csv("fully_preprocessed_dataset.csv", index=False)'''

In [ ]:
# Exploratory Data Analysis (EDA)

plt.figure(figsize=(10,6))
sns.heatmap(merged_df.corr(numeric_only=True), annot=True, cmap="coolwarm")
plt.title("Feature Correlation Matrix")
plt.show()

In [ ]:
# Training Multiple Baseline Models (Trial and Error)

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "SVM": SVC(),
    "KNN": KNeighborsClassifier(),
    "Random Forest": RandomForestClassifier()
}

results = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    print(name, "Accuracy:", acc)
    results[name] = acc
    
    # saving each model with its name
    filename = name.lower().replace(" ", "_")
    joblib.dump(model, f"test_model_{filename}.pkl")

print("\nAll models saved:")
for name, acc in results.items():
    print(f"  test_model_{name.lower().replace(' ', '_')}.pkl  —  {acc:.4f}")